# Coachable Robots: Edge-to-Cloud Training Pipeline

Benchmarking SO-ARM101 policy training with LeRobot on Chameleon Cloud MI100 GPUs.

## Architecture

```
┌─────────────────────┐     HuggingFace Hub      ┌──────────────────────────┐
│   Raspberry Pi 5    │  ──── dataset push ────>  │   Chameleon MI100 Node   │
│   (Edge Collector)  │                           │   (Training Server)      │
│                     │                           │                          │
│  xbox_soarm_teleop  │                           │  LeRobot + PyTorch ROCm  │
│  + 2x webcams       │                           │  ACT / Diffusion / Pi0   │
│  + SO-ARM101        │                           │                          │
│                     │  <── checkpoint pull ───  │  Trained policy ckpt     │
│  Docker container   │                           │                          │
│  (lerobot + teleop) │                           │  ROCm 6.3 + gfx908      │
└─────────────────────┘                           └──────────────────────────┘
```

## Notebook Goals

1. **Idempotent provisioning** — check for existing leases/servers before creating
2. **MI100 training environment** — ROCm + LeRobot on Chameleon bare metal
3. **Pi edge integration** — Docker container that collects, pushes, and fetches
4. **Benchmarking** — inference latency across hardware tiers

---
## Part 1: Chameleon Cloud Setup

In [1]:
import chi
from chi import lease, server, hardware
from chi.lease import Lease
from datetime import timedelta

# === CONFIGURE: Set your site, project, and resource names ===
chi.use_site("CHI@TACC")          # CHI@TACC or CHI@UC
chi.set("project_name", "CHI-XXXXXX")  # Your Chameleon allocation

# === Configuration ===
LEASE_NAME = "coachable-robots-mi100"
SERVER_NAME = "coachable-robots-training"
KEY_NAME = "YOUR_KEY_NAME"        # Key pair registered with Nova
NODE_TYPE = "gpu_mi100"
IMAGE_NAME = "CC-Ubuntu22.04"
LEASE_HOURS = 6

Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org


### 1a. Inspect Resources and Get/Create Lease

Checks for existing leases and servers. If a matching lease is found,
reuses it. If not, offers to create one. Never spawns duplicates.

In [2]:
my_lease = None

# ── Check for existing leases ──
print("=" * 60)
print("EXISTING LEASES")
print("=" * 60)

all_leases = lease.list_leases()
active_leases = [l for l in all_leases if l.status in ("ACTIVE", "PENDING")]

if active_leases:
    for i, l in enumerate(active_leases):
        marker = " <<<" if l.name == LEASE_NAME else ""
        print(f"  {i+1}. [{l.status}] {l.name}  (id: {l.id}){marker}")
        print(f"              ends: {l.end_date}")
        if l.node_reservations:
            print(f"              nodes: {len(l.node_reservations)} reservation(s)")
        if l.fip_reservations:
            print(f"              fips:  {len(l.fip_reservations)} reservation(s)")
        print()
        # Auto-select our named lease if it exists
        if l.name == LEASE_NAME:
            my_lease = l
else:
    print("  No active or pending leases.")

# ── Check for existing servers ──
print("=" * 60)
print("EXISTING SERVERS")
print("=" * 60)

try:
    existing_servers = server.list_servers()
    if existing_servers:
        for s in existing_servers:
            s_name = s.name if hasattr(s, 'name') else s.get('name', 'unknown')
            s_status = s.status if hasattr(s, 'status') else s.get('status', 'unknown')
            s_id = s.id if hasattr(s, 'id') else s.get('id', 'unknown')
            marker = " <<<" if s_name == SERVER_NAME else ""
            print(f"  [{s_status}] {s_name}  (id: {s_id}){marker}")
    else:
        print("  No servers running.")
except Exception as e:
    print(f"  Could not list servers: {e}")

# ── Check MI100 availability ──
print()
print("=" * 60)
print("MI100 AVAILABILITY")
print("=" * 60)
available_nodes = hardware.get_nodes(node_type=NODE_TYPE, filter_reserved=True)
print(f"  {len(available_nodes)} '{NODE_TYPE}' node(s) available for reservation.")

# ── Decision ──
print()
print("=" * 60)
if my_lease:
    print(f"REUSING lease '{my_lease.name}' [{my_lease.status}]")
    print(f"  ID:  {my_lease.id}")
    print(f"  End: {my_lease.end_date}")
else:
    print(f"No active lease named '{LEASE_NAME}'.")
    if not available_nodes:
        print(f"  WARNING: No {NODE_TYPE} nodes available either.")
        print("  Check the host calendar or try a different node type.")
    else:
        resp = input(f"Create a {LEASE_HOURS}h lease for 1x {NODE_TYPE}? [y/N]: ")
        if resp.strip().lower() in ('y', 'yes'):
            my_lease = Lease(
                name=LEASE_NAME,
                duration=timedelta(hours=LEASE_HOURS),
            )
            my_lease.add_node_reservation(node_type=NODE_TYPE, amount=1)
            my_lease.add_fip_reservation(amount=1)
            my_lease.submit(
                wait_for_active=True,
                wait_timeout=600,
                show="widget",
                idempotent=True,
            )
            print(f"\nLease ACTIVE: {my_lease.id}")
        else:
            print("Skipped lease creation. Re-run this cell when ready.")

EXISTING LEASES


Unauthorized: The request you have made requires authentication. (HTTP 401) (Request-ID: req-ece53eea-8f3a-4d8e-8a01-3c87bceaa62b)

In [4]:
import chi                                                                                                     
for site in ["CHI@TACC", "CHI@UC", "CHI@Edge"]:           
  try:                                                                                                       
      chi.use_site(site)
      from chi import lease                                                                                  
      lease.list_leases()                               
      print(f"{site}: ✅ auth OK")
  except Exception as e:                                                                                     
      print(f"{site}: ❌ {str(e)[:60]}")
                                               

Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org
CHI@TACC: ❌ The request you have made requires authentication. (HTTP 401
Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org
CHI@UC: ❌ The request you have made requires authentication. (HTTP 401
Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org
CHI@Edge: ❌ The request you have made requires authentication. (HTTP 401


### 1b. Create or Reuse Server

In [ ]:
import socket
import time

if my_lease is None:
    raise RuntimeError("No lease available. Run the previous cell and create one first.")

# Check if server already exists
gpu_server = None
floating_ip = None

try:
    existing_id = server.get_server_id(SERVER_NAME)
    gpu_server_obj = server.get_server(existing_id)
    status = gpu_server_obj.status if hasattr(gpu_server_obj, 'status') else gpu_server_obj.get('status')
    
    if status == "ACTIVE":
        print(f"Server '{SERVER_NAME}' already exists and is ACTIVE.")
        gpu_server = gpu_server_obj
    elif status == "BUILD":
        print(f"Server '{SERVER_NAME}' is still building. Waiting...")
        server.wait_for_active(existing_id)
        gpu_server = server.get_server(existing_id)
    else:
        print(f"Server '{SERVER_NAME}' exists but status is {status}. Deleting and recreating...")
        server.delete_server(existing_id)
        time.sleep(10)
except Exception:
    print(f"No existing server '{SERVER_NAME}'. Will create.")

# Create server if needed
if gpu_server is None:
    reservation_id = my_lease.node_reservations[0]["id"]
    print(f"Creating server with reservation {reservation_id}...")
    gpu_server = server.create_server(
        SERVER_NAME,
        reservation_id=reservation_id,
        image_name=IMAGE_NAME,
        key_name=KEY_NAME,
    )
    server.wait_for_active(gpu_server.id)
    print("Server is ACTIVE!")

# Attach or find floating IP
try:
    # Check if server already has a floating IP
    ips = server.list_floating_ips(gpu_server.id) if hasattr(server, 'list_floating_ips') else []
    if ips:
        floating_ip = ips[0]
        print(f"Existing floating IP: {floating_ip}")
except Exception:
    pass

if not floating_ip:
    floating_ip = server.associate_floating_ip(gpu_server.id)
    print(f"Assigned floating IP: {floating_ip}")

print(f"\nSSH: ssh cc@{floating_ip}")

### 1c. Wait for SSH and Verify MI100

In [ ]:
print(f"Waiting for SSH on {floating_ip}...")
timeout = 300
start = time.perf_counter()

while True:
    try:
        with socket.create_connection((floating_ip, 22), timeout=10):
            print("SSH ready!")
            break
    except OSError:
        elapsed = time.perf_counter() - start
        if elapsed >= timeout:
            print(f"Timed out after {timeout}s.")
            break
        print(f"  {elapsed:.0f}s... retrying")
        time.sleep(15)

In [ ]:
from chi import ssh

# MI100 is AMD — use lspci and rocm-smi, NOT nvidia-smi
with ssh.Remote(floating_ip) as conn:
    print("=== OS ===")
    conn.run("lsb_release -ds")
    
    print("\n=== AMD GPU Detection ===")
    conn.run("lspci | grep -i 'display\|vga\|amd\|radeon\|arcturus'")
    
    print("\n=== Kernel ===")
    conn.run("uname -r")
    
    print("\n=== Memory ===")
    conn.run("free -h | head -2")

---
## Part 2: Configure Node with Ansible

Uses an Ansible playbook to install ROCm + LeRobot. Idempotent —
safe to re-run if interrupted or if the node already has partial setup.

The playbook lives in `ansible/playbooks/setup_training_node.yml` and installs:
1. System deps (build tools, headers, tmux)
2. ROCm 6.3 (skips if already installed)
3. Miniconda (skips if already installed)
4. PyTorch 2.7.1 + ROCm 6.3 in a `lerobot` conda env (Python 3.12)
5. LeRobot v0.5.0 + HuggingFace CLI (`hf`)

In [ ]:
import subprocess, os

# Path to the ansible directory in your Chameleon Jupyter workspace
ANSIBLE_DIR = "/work/projects/coachable-robots/ansible"
PRIVATE_KEY = "/work/.ssh/id_rsa"
VAULT_PASSWORD_FILE = os.path.join(ANSIBLE_DIR, ".vault_pass")
# Create with: echo 'yourpassword' > ansible/.vault_pass && chmod 600 ansible/.vault_pass

# Generate inventory file with the current floating IP
inventory_content = f"""[training]
mi100 ansible_host={floating_ip} ansible_user=cc ansible_ssh_private_key_file={PRIVATE_KEY}

[training:vars]
ansible_ssh_common_args=-o StrictHostKeyChecking=no
"""

inventory_path = os.path.join(ANSIBLE_DIR, "inventory.ini")
os.makedirs(ANSIBLE_DIR, exist_ok=True)
with open(inventory_path, "w") as f:
    f.write(inventory_content)

print(f"Inventory written to {inventory_path}")
print(f"  Target: cc@{floating_ip}")
print(f"  Key:    {PRIVATE_KEY}")
print()
print("Run the next cell to execute the playbook, or run manually:")
print(f"  cd {ANSIBLE_DIR}")
print(f"  ansible-playbook -i inventory.ini playbooks/setup_training_node.yml --vault-password-file .vault_pass")

In [ ]:
# Run the Ansible playbook (15-20 min on first run, fast on re-runs)
# Streams output in real time so you can watch progress.

playbook_path = os.path.join(ANSIBLE_DIR, "playbooks/setup_training_node.yml")

cmd = [
    "ansible-playbook",
    "-i", inventory_path,
    playbook_path,
    "--vault-password-file", VAULT_PASSWORD_FILE,
    "-v",  # verbose — remove for less output
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=ANSIBLE_DIR,
)

for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print("=" * 60)
if rc == 0:
    print("Playbook completed successfully.")
else:
    print(f"Playbook failed with exit code {rc}.")
    print("Re-run this cell to retry — it will skip completed steps.")

---
## Part 3: Raspberry Pi Edge — Coaching Workflow

The Pi runs a Docker container (`rianders/lerobot-soarm101:latest`) that wraps LeRobot
with the `coachable` CLI. This part covers the full coaching workflow:

1. Verify devices (ports, camera)
2. Camera preview
3. Calibrate arms
4. Teleoperate (verify leader/follower)
5. Collect demonstration episodes
6. Replay to verify
7. Push dataset to HuggingFace Hub


In [ ]:
# === CONFIGURE ===
PI_IP = "192.168.4.191"
PI_SSH_PORT = 22222
PI_USER = "root"
PI_IMAGE = "rianders/lerobot-soarm101:latest"
HF_ORG = "ricklon"

# Serial numbers (confirmed 2026-04-06):
#   ttyACM0 = leader  (serial 5970072696, free-moving, no torque)
#   ttyACM1 = follower (serial 5970072616, torque enabled)
# NOTE: ports can swap on reconnect — verify with Step 3b before collecting.

import subprocess, shlex, time

def pi_ssh(cmd, capture=False):
    """Run a command on the Pi over SSH."""
    ssh_cmd = f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_IP}"
    full = shlex.split(ssh_cmd) + ["bash", "-c", cmd]
    if capture:
        return subprocess.check_output(full, text=True).strip()
    else:
        subprocess.run(full, check=True)

def pi_balena(args, interactive=False, devices=None, volumes=None, env=None):
    """Run a balena container on the Pi."""
    flags = "-it" if interactive else "-i"
    dev_flags = " ".join(f"--device={d}" for d in (devices or []))
    vol_flags = " ".join(f"-v {v}" for v in (volumes or []))
    env_flags = " ".join(f"-e {e}" for e in (env or []))
    return f"balena run {flags} --privileged {dev_flags} {vol_flags} {env_flags} {PI_IMAGE}"

print("Pi config set.")
print(f"  Host: {PI_USER}@{PI_IP}:{PI_SSH_PORT}")
print(f"  Image: {PI_IMAGE}")


### 3a. Verify Devices

In [ ]:
# Check both serial ports and camera are present
print("=== Serial Ports ===")
pi_ssh("ls /dev/ttyACM* 2>/dev/null || echo 'NO SERIAL PORTS FOUND'")

print("\n=== Camera ===")
pi_ssh("dmesg | grep -i 'uvc\|C920\|webcam' | tail -3 || echo 'No UVC camera in dmesg'")
pi_ssh("ls /dev/video0 2>/dev/null && echo 'video0 OK' || echo 'video0 NOT FOUND'")

print("\n=== Running Containers ===")
pi_ssh("balena ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'")


### 3b. Identify Leader/Follower Ports

The leader arm (free-moving, no torque) and follower arm (stiff, torque enabled) can
swap `ttyACM` numbers on reconnect. Run this cell to confirm which is which.

**Move the arm on ttyACM0 when prompted.** If values change → ttyACM0 is leader.
If flat → ttyACM0 is follower (swap ports in fleet.yaml).

Known serials:
- Leader:   `5970072696` 
- Follower: `5970072616`


In [ ]:
# Read positions from ttyACM0 with countdown — move the arm when prompted
check_cmd = '''
import time
from lerobot.motors.feetech.feetech import FeetechMotorsBus
from lerobot.motors import Motor, MotorNormMode
motors = {str(i): Motor(i, "sts3215", MotorNormMode.RANGE_M100_100) for i in range(1,7)}
bus = FeetechMotorsBus(port="/dev/ttyACM0", motors=motors)
bus.connect()
print("Get ready to move ttyACM0 arm...")
for i in [3,2,1]: print(f"{i}..."); __import__("time").sleep(1)
print("MOVE ttyACM0 NOW")
for _ in range(10):
    pos = bus.sync_read("Present_Position", normalize=False)
    print(list(pos.values()))
    time.sleep(0.5)
bus.disconnect()
'''

pi_ssh(f"balena run --privileged --device=/dev/ttyACM0 {PI_IMAGE} python3 -c '{check_cmd}'")


### 3c. Camera Preview

In [ ]:
# Stop any existing preview container, start fresh
import subprocess, shlex, time

PREVIEW_PORT = 7860

print("Stopping existing preview container...")
pi_ssh("balena stop $(balena ps -q --filter ancestor=rianders/lerobot-soarm101) 2>/dev/null || true")
time.sleep(2)

print("Starting camera preview...")
pi_ssh(
    f"balena run -d --privileged --device=/dev/video0 --device=/dev/video1 "
    f"-p {PREVIEW_PORT}:{PREVIEW_PORT} {PI_IMAGE} "
    f"python scripts/camera_preview.py"
)
time.sleep(3)

# Open SSH tunnel
tunnel = subprocess.Popen(
    shlex.split(f"ssh -p {PI_SSH_PORT} -L {PREVIEW_PORT}:localhost:{PREVIEW_PORT} -N -o StrictHostKeyChecking=no {PI_USER}@{PI_IP}")
)
time.sleep(2)
print(f"Preview tunnel open. View at: http://localhost:{PREVIEW_PORT}")


In [ ]:
from IPython.display import IFrame, display
display(IFrame(src=f"http://localhost:{PREVIEW_PORT}", width="100%", height=500))


### 3d. Calibrate Arms

Run calibration once per arm setup. Files saved to `/mnt/data/calibration` on the Pi
and persist across container restarts.

**During calibration:**
- Move each joint slowly to its min and max limits
- Don't forget the gripper — open and close fully
- The gripper cable (motor 6) can unplug during wrist rotation — check if calibration fails

**Only re-calibrate if:** a servo is replaced, arm is reassembled, or calibration data is corrupt.


In [ ]:
# Stop camera preview first (can't share camera during calibration)
pi_ssh("balena stop $(balena ps -q) 2>/dev/null || true")
time.sleep(2)

# Copy fleet config to Pi
subprocess.run(shlex.split(
    f"scp -P {PI_SSH_PORT} config/fleet.yaml {PI_USER}@{PI_IP}:/tmp/fleet.yaml"
), check=True)

print("Starting calibration...")
print("IMPORTANT: Move each joint to min AND max when prompted.")
print("           Don't forget the gripper (open and close fully).")
print()
print(f"Run in your terminal:")
print()
print(f"ssh -p {PI_SSH_PORT} -t {PI_USER}@{PI_IP} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   {PI_IMAGE} \\')
print(f'   coachable --fleet /app/config/fleet.yaml calibrate --robot alpha"')


### 3e. Teleoperate (Verify Arms)

Before collecting data, verify teleoperation works — the follower should smoothly
track the leader in real time at 60Hz.

**What you'll see:** The follower arm snaps to match the leader on connect, then
tracks continuously. `Ctrl+C` to stop (torque releases cleanly).


In [ ]:
print("Run in your terminal to verify teleoperation:")
print()
print(f"ssh -p {PI_SSH_PORT} -t {PI_USER}@{PI_IP} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-teleoperate \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration --robot.id=alpha_follower \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration --teleop.id=alpha_leader"')


### 3f. Collect Demonstration Episodes

Coach the robot by demonstrating the task with the leader arm.

**Timing:**
- Config dump appears first (~5 seconds) — **ignore it**
- Wait for `Recording episode N` — **that's your cue to start moving**
- Move the leader arm continuously for the full episode duration
- During RESET time: return leader arm to home position

**Tips:**
- Start with 3-5 short episodes (15s) to verify data quality
- Check data with the replay cell before collecting a full dataset
- Collect 50-100 episodes for a trainable policy


In [ ]:
# === CONFIGURE your collection run ===
ROBOT = "alpha"
DATASET_SLUG = "pick_block"       # task name — becomes ricklon/soarm101-pick_block
TASK_DESC = "Pick up the block and place it in the bowl"
NUM_EPISODES = 50
EPISODE_TIME = 30   # seconds per episode
RESET_TIME = 10     # seconds to return to home between episodes

DATASET_REPO = f"{HF_ORG}/soarm101-{DATASET_SLUG}"

print(f"Dataset:  {DATASET_REPO}")
print(f"Episodes: {NUM_EPISODES} x {EPISODE_TIME}s + {RESET_TIME}s reset")
total = NUM_EPISODES * (EPISODE_TIME + RESET_TIME) - RESET_TIME
print(f"Total:    ~{total}s ({total//60}m {total%60}s)")
print()
print("Run in your terminal:")
print()
print(f"ssh -p {PI_SSH_PORT} -t {PI_USER}@{PI_IP} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video1 \\')
print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   coachable --fleet /app/config/fleet.yaml collect \\')
print(f'     --robot {ROBOT} --dataset {DATASET_SLUG} \\')
print(f'     --episodes {NUM_EPISODES} --episode-time {EPISODE_TIME} --reset-time {RESET_TIME} \\')
print(f"     --task '{TASK_DESC}' --no-push"  + '"')


### 3g. Replay Episode (Verify Data)

In [ ]:
# Replay episode 0 to verify data quality before pushing to Hub
EPISODE = 0

print(f"Replaying episode {EPISODE} from {DATASET_REPO}")
print("Watch the follower arm — it should reproduce your demonstration.")
print()
print("Run in your terminal:")
print()
print(f"ssh -p {PI_SSH_PORT} -t {PI_USER}@{PI_IP} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-replay \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.id=alpha_follower --robot.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={DATASET_REPO} \\')
print(f'     --dataset.root=/app/data/{HF_ORG}/soarm101-{DATASET_SLUG} \\')
print(f'     --dataset.episode={EPISODE} --play_sounds=false"')


### 3h. Push Dataset to HuggingFace Hub

In [ ]:
# Push the dataset to HuggingFace Hub using the push script.
# Token is read securely via prompt (not stored in notebook).
import subprocess
print("Pushing dataset to HuggingFace Hub...")
print("You will be prompted for your HF write token.")
print()
subprocess.run(["bash", "scripts/push_dataset.sh", DATASET_SLUG], check=True)
print(f"\nDataset live at: https://huggingface.co/datasets/{DATASET_REPO}")


---
## Part 4: Training on MI100

Once the Pi has pushed a dataset, trigger training on the Chameleon node.

**MI100 (gfx908) constraints:**
- No Flash Attention 2 (gfx90a+ only)
- No hipBLASLt
- 32 GB HBM2 — plenty for ACT, tight for Pi0 3B
- ACT is recommended for MI100; Pi0 needs gradient checkpointing + bf16

In [ ]:
# Trigger training on the remote MI100 node
# Replace HF_USER and dataset name with your values

HF_USER = "YOUR_HF_USERNAME"  # CONFIGURE: your HuggingFace username
DATASET = "soarm101-pick_block"  # Dataset pushed from Pi
POLICY = "act"  # 'act' for MI100, 'pi0' for H100

train_cmd = f"""
source ~/miniconda3/bin/activate lerobot
cd ~/lerobot

python lerobot/scripts/train.py \\
    --dataset.repo_id={HF_USER}/{DATASET} \\
    --policy.path=lerobot/{POLICY} \\
    --output_dir=outputs/train/{POLICY}_{DATASET} \\
    --job_name={POLICY}_{DATASET} \\
    --policy.device=cuda \\
    --wandb.enable=false
"""

print("Training command (run on MI100 node):")
print(train_cmd)
print(f"To execute: ssh cc@{floating_ip} and paste the above")
print("Or run the next cell to launch it remotely.")

In [ ]:
# Optional: launch training remotely (runs in foreground — long!)
# Consider using tmux or nohup via SSH for production runs.
#
# with ssh.Remote(floating_ip) as conn:
#     conn.run(train_cmd)

---
## Part 5: Benchmark Inference Latency

For coachable-robots-bench, record inference timing on each tier.

In [ ]:
benchmark_script = """
import torch, time, json, platform

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device_name = torch.cuda.get_device_name(0) if device == 'cuda' else platform.processor()

results = {'device': device_name, 'pytorch': torch.__version__, 'tests': []}

for label, shape in [('policy_input_480p', (1, 3, 480, 640)), ('policy_input_224', (1, 3, 224, 224))]:
    x = torch.randn(*shape, device=device)
    # Warmup
    for _ in range(20):
        _ = torch.nn.functional.interpolate(x, size=(224, 224), mode='bilinear')
    if device == 'cuda': torch.cuda.synchronize()
    
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        _ = torch.nn.functional.interpolate(x, size=(224, 224), mode='bilinear')
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    
    results['tests'].append({
        'name': label, 'shape': list(shape),
        'avg_ms': round(sum(times)/len(times), 3),
        'p50_ms': round(sorted(times)[len(times)//2], 3),
        'p99_ms': round(sorted(times)[int(len(times)*0.99)], 3),
    })

print(json.dumps(results, indent=2))
"""

with ssh.Remote(floating_ip) as conn:
    conn.run(f"""
        source ~/miniconda3/bin/activate lerobot
        python3 -c '{benchmark_script}'
    """)

---
## Part 6: Cleanup

**Run this when done** to release bare-metal resources and stop burning SUs.

In [ ]:
# Safety: show what we're about to delete
print("Will delete:")
print(f"  Server: {SERVER_NAME}")
print(f"  Lease:  {LEASE_NAME} (id: {my_lease.id})")
print()
confirm = input("Type 'yes' to confirm cleanup: ")

if confirm.strip().lower() == 'yes':
    try:
        sid = server.get_server_id(SERVER_NAME)
        server.delete_server(sid)
        print(f"Server '{SERVER_NAME}' deleted.")
    except Exception as e:
        print(f"Server cleanup: {e}")

    try:
        lease.delete_lease(my_lease.id)
        print(f"Lease '{LEASE_NAME}' deleted. Hardware released.")
    except Exception as e:
        print(f"Lease cleanup: {e}")
    
    print("\nCleanup complete.")
else:
    print("Cleanup cancelled.")